## Cell 1 — Imports & config  (unchanged)

In [ ]:
import requests, urllib3
import pytz

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

file_date  = "2026-06-02"
url        = "<YOUR_ELASTICSEARCH_URL>/_search"
user       = "<USERNAME>"
psd        = "<PASSWORD>"
headers    = {"Content-Type": "application/json"}
all_scopes = ['SWCC', 'TDCC', 'PBCC', 'PTCC']

kolkata_tz      = pytz.timezone("Asia/Kolkata")
INTERVAL_MIN    = 10
total_intervals = (24 * 60) // INTERVAL_MIN   # 144

## Cell 2 — Fetch loop  (**CHANGED** — paginated `search_after`)

**Before:** single `_search` with `"size": 10000` → silently capped every interval at 10 000 records;
intervals that genuinely had fewer records (e.g. 9) could be real, but intervals with **more** than
10 000 records were silently truncated — the running total looked fine but data was missing.

**After:** uses `search_after` + `sort` to page through **all** records in each interval.
`track_total_hits: true` lets us verify the fetched count matches Elasticsearch's actual count
and emit a `[WARN]` if they ever diverge.

In [ ]:
all_hits = []

for i in range(total_intervals):
    start_total_min = i * INTERVAL_MIN
    end_total_min   = start_total_min + INTERVAL_MIN - 1

    start_hour, start_minute = divmod(start_total_min, 60)
    end_hour,   end_minute   = divmod(end_total_min,   60)

    start_time = f"{file_date}T{start_hour:02d}:{start_minute:02d}:00.000"
    end_time   = f"{file_date}T{end_hour:02d}:{end_minute:02d}:59.999"

    interval_hits  = []
    search_after   = None
    total_expected = None

    while True:
        body = {
            "track_total_hits": True,
            "size": 10000,
            "sort": [{"@timestamp": "asc"}, {"_id": "asc"}],
            "query": {
                "bool": {
                    "must": [
                        {"range": {"@timestamp": {"gte": start_time, "lte": end_time}}},
                        {"terms": {"scope.keyword": all_scopes}},
                    ]
                }
            },
        }
        if search_after:
            body["search_after"] = search_after

        response = requests.post(
            url,
            headers=headers,
            auth=(user, psd),
            json=body,
            verify=False,
            timeout=120,
        )
        if response.status_code != 200:
            raise Exception(
                f"API call failed for interval {start_time} - {end_time}: {response.text}"
            )

        resp_json      = response.json()
        hits_wrap      = resp_json.get("hits", {})
        total_expected = hits_wrap.get("total", {}).get("value", 0)
        hits           = hits_wrap.get("hits", [])

        if not hits:
            break

        interval_hits.extend(hits)
        search_after = hits[-1]["sort"]  # cursor for next page

        if len(interval_hits) >= total_expected:
            break

    if total_expected and len(interval_hits) != total_expected:
        print(
            f"  [WARN] interval {i+1}: ES reported {total_expected:,} hits "
            f"but fetched {len(interval_hits):,}"
        )

    all_hits.extend(interval_hits)
    print(
        f"Fetching interval {i+1}/{total_intervals}: {start_time} to {end_time}  "
        f"|  interval records: {len(interval_hits):,} | running total: {len(all_hits):,}"
    )

print(f"\nAll {total_intervals} intervals fetched. Total records: {len(all_hits):,}")


In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

WINDOW_MIN = 30  # minutes per window

# Group all_hits by 30-min window using @timestamp
windows = {}
for record in all_hits:
    fields  = record.get("fields", {})
    ts_raw  = fields.get("@timestamp", [None])
    ts_str  = ts_raw[0] if isinstance(ts_raw, list) else ts_raw
    if ts_str:
        dt      = datetime.fromisoformat(ts_str[:16])
        win_key = (dt.hour * 60 + dt.minute) // WINDOW_MIN   # 0–47
    else:
        win_key = -1
    windows.setdefault(win_key, []).append(record)

total_windows = 24 * 60 // WINDOW_MIN   # 48

# Build one DF per window, union them — driver serialises ~19k records at a time (not 9 lakh).
# Union is lazy in Spark, so the final df is processed in chunks — no OOM.
# df is available after the loop so display(df) works normally.
df = None

for win_key in sorted(windows.keys()):
    records   = windows[win_key]
    start_min = win_key * WINDOW_MIN
    h, m      = divmod(start_min, 60)
    print(f"Window {win_key + 1}/{total_windows}  [{h:02d}:{m:02d} – {h:02d}:{m + WINDOW_MIN - 1:02d}]:  {len(records):,} records")

    flattened_records = []
    for record in records:
        flat = {}
        for key, value in record["fields"].items():
            flat[key] = value[0] if isinstance(value, list) and len(value) > 0 else value
        flattened_records.append(flat)

    window_df = spark.createDataFrame(flattened_records)
    df = window_df if df is None else df.union(window_df)

    del window_df, flattened_records, records
    windows[win_key] = None

print(f"total columns: {len(df.columns)}")

## Cell 3 — DataFrame creation  (**CHANGED** — 30-minute windows)

**Before:** `spark.createDataFrame(all_flattened)` — loads all ~9 lakh records at once → driver OOM  
**After:** groups `all_hits` by 30-min window using `@timestamp`, creates one small DataFrame per window, then frees memory

In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

WINDOW_MIN = 30  # size of each window in minutes

# ── Group all_hits by 30-min window using @timestamp ─────────────────────────
windows = {}
for record in all_hits:
    fields  = record.get("fields", {})
    ts_raw  = fields.get("@timestamp", [None])
    ts_str  = ts_raw[0] if isinstance(ts_raw, list) else ts_raw
    if ts_str:
        dt      = datetime.fromisoformat(ts_str[:16])          # "2026-06-02T00:09"
        win_key = (dt.hour * 60 + dt.minute) // WINDOW_MIN     # 0 – 47
    else:
        win_key = -1
    windows.setdefault(win_key, []).append(record)

total_windows = 24 * 60 // WINDOW_MIN   # 48

# ── Process one 30-min window at a time ──────────────────────────────────────
for win_key in sorted(windows.keys()):
    records   = windows[win_key]
    start_min = win_key * WINDOW_MIN
    h, m      = divmod(start_min, 60)
    print(f"\nWindow {win_key + 1}/{total_windows}  [{h:02d}:{m:02d} – {h:02d}:{m + WINDOW_MIN - 1:02d}]:  {len(records):,} records")

    # flatten fields (same logic as before)
    flattened_records = []
    for record in records:
        flat = {}
        for key, value in record["fields"].items():
            flat[key] = value[0] if isinstance(value, list) and len(value) > 0 else value
        flattened_records.append(flat)

    df = spark.createDataFrame(flattened_records)
    print(f"  total columns: {len(df.columns)}")

    # ── add your transformations / writes here ────────────────────────────────
    # e.g. df.write.mode("append").parquet(f"/mnt/output/{file_date}/window_{win_key+1:02d}")
    # ─────────────────────────────────────────────────────────────────────────

    # free this window's memory before moving to the next
    del df, flattened_records, records
    windows[win_key] = None